In [1]:
# Imports
import os
import json
import pandas as pd
from openai import OpenAI
from tqdm.auto import tqdm  # Progress bar

In [2]:
# ============================================================
# CONFIGURATION
# ============================================================

API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
PROMPT_FILE = "failure_summarizer_layer2.txt"
CLUSTER_DIR = "./cluster_first_round"
OUTPUT_DIR = "results/cluster_analysis_layer2"
MAX_WORKERS = 20  # Adjust based on API rate limits

# Cluster selection - ADJUST THIS LINE
# Examples (uncomment one):
cluster_index = slice(None)        # All clusters
# cluster_index = slice(3, 4)      # Only cluster_0.json
# cluster_index = slice(5, 10)     # cluster_5.json to cluster_9.json
# cluster_index = slice(None, 80)  # cluster_0.json to cluster_79.json
# cluster_index = slice(10, 20)    # cluster_10.json to cluster_19.json

In [3]:
# Load cluster files
import glob

all_cluster_files = sorted(glob.glob(os.path.join(CLUSTER_DIR, "cluster_*.json")), 
                           key=lambda x: int(os.path.basename(x).replace('cluster_', '').replace('.json', '')))

# Apply cluster selection
cluster_files = all_cluster_files[cluster_index]
print(f"✓ Found {len(all_cluster_files)} total cluster files")
print(f"✓ Selected {len(cluster_files)} clusters to process")

# Load selected clusters
clusters = []
for file_path in cluster_files:
    with open(file_path, 'r', encoding='utf-8') as f:
        cluster_data = json.load(f)
        cluster_label = int(os.path.basename(file_path).replace('cluster_', '').replace('.json', ''))
        clusters.append({
            'cluster_label': cluster_label,
            'file_path': file_path,
            'data': cluster_data,
            'size': len(cluster_data)
        })

print(f"✓ Loaded {len(clusters)} clusters")
if len(clusters) > 0:
    print(f"  Cluster range: {min(c['cluster_label'] for c in clusters)} - {max(c['cluster_label'] for c in clusters)}")
    print(f"  Size range: {min(c['size'] for c in clusters)} - {max(c['size'] for c in clusters)} items")

✓ Found 80 total cluster files
✓ Selected 80 clusters to process
✓ Loaded 80 clusters
  Cluster range: 0 - 79
  Size range: 1 - 34 items


In [4]:
# Load prompt template
with open(PROMPT_FILE, 'r', encoding='utf-8') as f:
    prompt_template = f.read()

print(prompt_template)

You are summarizing a cluster of reasons why an AI model's answer to Theory of Mind questions differs from the human answer. Each reason corresponds to one question.

**Task:**

Carefully read each reason and analyze step by step how the reasons in this cluster share common patterns. Then summarize your analysis in exactly 3 sentences and 90+-10 words.

**Reasons in this cluster:**

{reasons_list}



In [5]:
# Initialize OpenAI client
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=API_KEY
)

def call_gpt51_summarizer(client, prompt):
    """Call Gemini-3-Pro without reasoning"""
    response = client.chat.completions.create(
        model="openai/gpt-5.1",
        messages=[{"role": "user", "content": prompt}], 
        temperature=0.0,
        max_tokens=10240
    )
    # print(response)
    message = response.choices[0].message
    content = message.content 
    print(content)
    
    return content

print("✓ Client initialized")

✓ Client initialized


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def process_single_cluster(cluster_info):
    """Process a single cluster"""
    cluster_label = cluster_info['cluster_label']
    cluster_data = cluster_info['data']
    
    # Create thread-specific client
    thread_client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=API_KEY
    )
    
    # Build reasons list from all items in cluster
    reasons_list = []
    for i, item in enumerate(cluster_data, 1):
        reason_text = f"{i}. {item['failed_summary']}"
        reasons_list.append(reason_text)
    
    # Format prompt with all reasons
    reasons_text = "\n\n".join(reasons_list)
    prompt = prompt_template.format(reasons_list=reasons_text)
    
    try:
        cluster_summary = call_gpt51_summarizer(thread_client, prompt)
        
        result = {
            'cluster_label': cluster_label,
            'cluster_size': len(cluster_data),
            'cluster_summary': cluster_summary,
            'reasons': [item['failed_summary'] for item in cluster_data],
            'original_indices': [item['original_index'] for item in cluster_data]
        }
        return cluster_label, result, None
        
    except Exception as e:
        print(f"\n✗ Error at cluster {cluster_label}: {str(e)[:100]}")
        result = {
            'cluster_label': cluster_label,
            'cluster_size': len(cluster_data),
            'cluster_summary': f"ERROR: {str(e)}",
            'reasons': [item['failed_summary'] for item in cluster_data],
            'original_indices': [item['original_index'] for item in cluster_data]
        }
        return cluster_label, result, str(e)

# Process clusters with multithreading
results = []

print(f"Processing {len(clusters)} clusters with {MAX_WORKERS} threads...")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # Submit all tasks
    future_to_cluster = {
        executor.submit(process_single_cluster, cluster): cluster['cluster_label']
        for cluster in clusters
    }
    
    # Process completed tasks with progress bar
    for future in tqdm(as_completed(future_to_cluster), 
                      total=len(clusters), 
                      desc="Processing clusters"):
        cluster_label, result, error = future.result()
        results.append(result)

print(f"\n✓ Completed processing {len(results)} clusters")

In [29]:
# Convert to DataFrame and sort by cluster label
results_df = pd.DataFrame(results)

# Sort by cluster label to maintain order
if 'cluster_label' in results_df.columns:
    results_df = results_df.sort_values('cluster_label').reset_index(drop=True)

print(f"Results: {results_df.shape}")
print(f"\nWith summary: {results_df['cluster_summary'].notna().sum()}/{len(results_df)}")
print(f"Errors: {results_df['cluster_summary'].str.startswith('ERROR').sum()}")

Results: (80, 5)

With summary: 80/80
Errors: 0


In [30]:
results_df

,cluster_label,cluster_size,cluster_summary,reasons,original_indices
0,0,2,"Across these reasons, the model relies on a si...",[The model likely focused on the surface facts...,"[1091, 1918]"
1,1,1,This cluster reflects a pattern where the mode...,[The model likely reasoned that Wang Lei’s exc...,[1302]
2,2,2,"Across these reasons, the core pattern is how ...",[The story states there are 50 lunches in tota...,"[165, 1923]"
3,3,7,"Across these items, the model repeatedly relie...",[The model likely misapplied a common “false-b...,"[461, 527, 774, 839, 900, 991, 1859]"
4,4,10,"Across these items, the model repeatedly privi...",[The model likely focused on the social implic...,"[261, 450, 806, 932, 1047, 1425, 1445, 1562, 1..."
...,...,...,...,...,...
75,75,14,"Across these cases, the model repeatedly over-...",[The model likely focused on a familiar narrat...,"[57, 542, 656, 850, 866, 930, 1080, 1138, 1170..."
76,76,2,"Across these reasons, the model consistently g...","[From the story, Xiao Lei explicitly praises “...","[1694, 1973]"
77,77,3,"Across these items, the model systematically t...",[The model likely focused on the emotional asp...,"[882, 1141, 1773]"
78,78,6,"Across these cases, the model relies on shallo...",[Xiao Hong skips her club duty to visit a frie...,"[322, 510, 961, 975, 1042, 1695]"


In [36]:
# Statistics
print(f"Total clusters analyzed: {len(results_df)}")
print(f"With summaries: {results_df['cluster_summary'].notna().sum()}")
print(f"With errors: {results_df['cluster_summary'].str.startswith('ERROR').sum()}")
print(f"Total failures covered: {results_df['cluster_size'].sum()}")
print(f"Avg cluster size: {results_df['cluster_size'].mean():.1f}")
print(f"Cluster size range: {results_df['cluster_size'].min()} - {results_df['cluster_size'].max()}")
# Compute word counts
word_counts = results_df['cluster_summary'].str.split().str.len()
print(f"Avg summary length: {word_counts.mean():.1f} words")
print(f"Max summary length: {word_counts.max()} words")
print(f"Min summary length: {word_counts.min()} words")

Total clusters analyzed: 80
With summaries: 80
With errors: 0
Total failures covered: 645
Avg cluster size: 8.1
Cluster size range: 1 - 34
Avg summary length: 88.3 words
Max summary length: 100 words
Min summary length: 80 words


In [37]:
# Save results - one file per cluster
os.makedirs(OUTPUT_DIR, exist_ok=True)

saved_files = []
for result in tqdm(results, desc="Saving cluster files"):
    cluster_label = result['cluster_label']
    
    # Create individual cluster output
    cluster_output = {
        'cluster_label': cluster_label,
        'cluster_size': result['cluster_size'],
        'cluster_summary': result['cluster_summary'],
        'reasons': result['reasons'],
        'original_indices': result['original_indices']
    }
    
    # Save as JSON
    output_file = os.path.join(OUTPUT_DIR, f"cluster_{cluster_label}_analysis.json")
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(cluster_output, f, indent=2, ensure_ascii=False)
    
    saved_files.append(output_file)

# Also save summary CSV for overview
summary_df_save = results_df[['cluster_label', 'cluster_size', 'cluster_summary']].copy()
summary_csv = os.path.join(OUTPUT_DIR, f"cluster_summaries_overview.csv")
summary_df_save.to_csv(summary_csv, index=False)

print(f"\n✓ Saved {len(saved_files)} individual cluster files to: {OUTPUT_DIR}")
print(f"  Files: cluster_0_analysis.json, cluster_1_analysis.json, ...")
print(f"\n✓ Saved summary overview to: {summary_csv}")

Saving cluster files:   0%|          | 0/80 [00:00<?, ?it/s]


✓ Saved 80 individual cluster files to: results/cluster_analysis_layer2
  Files: cluster_0_analysis.json, cluster_1_analysis.json, ...

✓ Saved summary overview to: results/cluster_analysis_layer2/cluster_summaries_overview.csv
